In [35]:
!pip install psycopg2-binary pandas


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import psycopg2
import pandas as pd

db_params = {
    "host": "localhost",
    "port": 5432,
    "user": "postgres",
    "password": "123"
}

DB_NAME = "company_ceo_db"
conn = psycopg2.connect(database=DB_NAME, **db_params)
conn.autocommit = True
cur = conn.cursor()
cur.execute("""
DROP TABLE IF EXISTS Company CASCADE;
DROP TABLE IF EXISTS CEO CASCADE;
DROP TABLE IF EXISTS City CASCADE;
DROP TABLE IF EXISTS State CASCADE;
DROP TABLE IF EXISTS Country CASCADE;
""")
print("Dropped old tables.")
DB = "company_ceo_db"
DB_PARAMS = dict(host="localhost", port=5432, user="postgres", password="1234")
create_sql = """
CREATE TABLE Country (
    country_id   SERIAL PRIMARY KEY,
    country_name VARCHAR(50) UNIQUE NOT NULL
);

CREATE TABLE State (
    state_id     SERIAL PRIMARY KEY,
    state_name   VARCHAR(50),
    country_id   INT REFERENCES Country(country_id) ON DELETE CASCADE
);

CREATE TABLE City (
    city_id      SERIAL PRIMARY KEY,
    city_name    VARCHAR(50) UNIQUE NOT NULL,
    state_id     INT REFERENCES State(state_id)    ON DELETE SET NULL,
    country_id   INT REFERENCES Country(country_id) ON DELETE CASCADE
);

CREATE TABLE CEO (
    ceo_id       SERIAL PRIMARY KEY,
    first_name   VARCHAR(50),
    last_name    VARCHAR(50),
    birth_year   INT,
    birth_place  VARCHAR(100),
    start_year   INT
);

CREATE TABLE Company (
    company_id   SERIAL PRIMARY KEY,
    name         VARCHAR(100) UNIQUE NOT NULL,
    hq_city      INT REFERENCES City(city_id) ON DELETE SET NULL,
    ceo_id       INT REFERENCES CEO(ceo_id),
    founding_year INT NOT NULL
);
"""
with psycopg2.connect(database=DB, **DB_PARAMS) as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute(create_sql)
print("Created all tables.")

insert_data = """
INSERT INTO Country (country_name) VALUES
('USA'), ('China'), ('Japan'), ('South Korea'), ('France');


INSERT INTO State (state_name, country_id)
SELECT 'California', country_id FROM Country WHERE country_name = 'USA';
INSERT INTO State (state_name, country_id)
SELECT 'New York', country_id FROM Country WHERE country_name = 'USA';
INSERT INTO State (state_name, country_id)
SELECT 'Zhejiang', country_id FROM Country WHERE country_name = 'China';
INSERT INTO State (state_name, country_id)
SELECT 'Gyeonggi-do', country_id FROM Country WHERE country_name = 'South Korea';


INSERT INTO City (city_name, state_id, country_id)
SELECT 'Cupertino', s.state_id, c.country_id FROM State s
JOIN Country c ON s.country_id = c.country_id WHERE s.state_name = 'California';

INSERT INTO City (city_name, state_id, country_id)
SELECT 'Armonk', s.state_id, c.country_id FROM State s
JOIN Country c ON s.country_id = c.country_id WHERE s.state_name = 'New York';

INSERT INTO City (city_name, state_id, country_id)
SELECT 'Hangzhou', s.state_id, c.country_id FROM State s
JOIN Country c ON s.country_id = c.country_id WHERE s.state_name = 'Zhejiang';

INSERT INTO City (city_name, state_id, country_id)
SELECT 'Suwon', s.state_id, c.country_id FROM State s
JOIN Country c ON s.country_id = c.country_id WHERE s.state_name = 'Gyeonggi-do';

INSERT INTO City (city_name, state_id, country_id)
SELECT 'Paris', NULL, country_id FROM Country WHERE country_name = 'France';

INSERT INTO City (city_name, state_id, country_id)
SELECT 'Tokyo', NULL, country_id FROM Country WHERE country_name = 'Japan';


INSERT INTO CEO (first_name, last_name, birth_year, birth_place, start_year) VALUES
('Tim', 'Cook', 1960, 'Mobile, AL, USA', 2011),
('Arvind', 'Krishna', 1962, 'West Godavari, India', 2020),
('Andy', 'Jassy', 1968, 'Scarsdale, NY, USA', 2021),
('Mary', 'Barra', 1961, 'Royal Oak, MI, USA', 2014),
('Kenichiro', 'Yoshida', 1959, 'Kumamoto, Japan', 2018),
('Daniel', 'Zhang', 1972, 'Shanghai, China', 2015),
('Han', 'Jong-hee', 1962, 'Seoul, South Korea', 2022),
('François-Henri', 'Pinault', 1962, 'Rennes, France', 2005);


INSERT INTO Company (name, hq_city, ceo_id, founding_year)
SELECT 'Apple', city_id, 1, 1976 FROM City WHERE city_name = 'Cupertino';
INSERT INTO Company (name, hq_city, ceo_id, founding_year)
SELECT 'IBM', city_id, 2, 1911 FROM City WHERE city_name = 'Armonk';
INSERT INTO Company (name, hq_city, ceo_id, founding_year)
SELECT 'Sony', city_id, 5, 1946 FROM City WHERE city_name = 'Tokyo';
INSERT INTO Company (name, hq_city, ceo_id, founding_year)
SELECT 'Alibaba', city_id, 6, 1999 FROM City WHERE city_name = 'Hangzhou';
INSERT INTO Company (name, hq_city, ceo_id, founding_year)
SELECT 'Samsung Electronics', city_id, 7, 1969 FROM City WHERE city_name = 'Suwon';
INSERT INTO Company (name, hq_city, ceo_id, founding_year)
SELECT 'Kering SA', city_id, 8, 1963 FROM City WHERE city_name = 'Paris';
"""
cur.execute(insert_data)
conn.commit()
print(" All seed data inserted successfully.")

OperationalError: connection to server at "localhost" (::1), port 5432 failed: FATAL:  password authentication failed for user "postgres"


In [44]:
insert_data = """
INSERT INTO Country (country_name) VALUES
('USA'), ('China'), ('Japan'), ('South Korea'), ('France');


INSERT INTO State (state_name, country_id)
SELECT 'California', country_id FROM Country WHERE country_name = 'USA';
INSERT INTO State (state_name, country_id)
SELECT 'New York', country_id FROM Country WHERE country_name = 'USA';
INSERT INTO State (state_name, country_id)
SELECT 'Zhejiang', country_id FROM Country WHERE country_name = 'China';
INSERT INTO State (state_name, country_id)
SELECT 'Gyeonggi-do', country_id FROM Country WHERE country_name = 'South Korea';


INSERT INTO City (city_name, state_id, country_id)
SELECT 'Cupertino', s.state_id, c.country_id FROM State s
JOIN Country c ON s.country_id = c.country_id WHERE s.state_name = 'California';

INSERT INTO City (city_name, state_id, country_id)
SELECT 'Armonk', s.state_id, c.country_id FROM State s
JOIN Country c ON s.country_id = c.country_id WHERE s.state_name = 'New York';

INSERT INTO City (city_name, state_id, country_id)
SELECT 'Hangzhou', s.state_id, c.country_id FROM State s
JOIN Country c ON s.country_id = c.country_id WHERE s.state_name = 'Zhejiang';

INSERT INTO City (city_name, state_id, country_id)
SELECT 'Suwon', s.state_id, c.country_id FROM State s
JOIN Country c ON s.country_id = c.country_id WHERE s.state_name = 'Gyeonggi-do';

INSERT INTO City (city_name, state_id, country_id)
SELECT 'Paris', NULL, country_id FROM Country WHERE country_name = 'France';

INSERT INTO City (city_name, state_id, country_id)
SELECT 'Tokyo', NULL, country_id FROM Country WHERE country_name = 'Japan';


INSERT INTO CEO (first_name, last_name, birth_year, birth_place, start_year) VALUES
('Tim', 'Cook', 1960, 'Mobile, AL, USA', 2011),
('Arvind', 'Krishna', 1962, 'West Godavari, India', 2020),
('Andy', 'Jassy', 1968, 'Scarsdale, NY, USA', 2021),
('Mary', 'Barra', 1961, 'Royal Oak, MI, USA', 2014),
('Kenichiro', 'Yoshida', 1959, 'Kumamoto, Japan', 2018),
('Daniel', 'Zhang', 1972, 'Shanghai, China', 2015),
('Han', 'Jong-hee', 1962, 'Seoul, South Korea', 2022),
('François-Henri', 'Pinault', 1962, 'Rennes, France', 2005);


INSERT INTO Company (name, hq_city, ceo_id, founding_year)
SELECT 'Apple', city_id, 1, 1976 FROM City WHERE city_name = 'Cupertino';
INSERT INTO Company (name, hq_city, ceo_id, founding_year)
SELECT 'IBM', city_id, 2, 1911 FROM City WHERE city_name = 'Armonk';
INSERT INTO Company (name, hq_city, ceo_id, founding_year)
SELECT 'Sony', city_id, 5, 1946 FROM City WHERE city_name = 'Tokyo';
INSERT INTO Company (name, hq_city, ceo_id, founding_year)
SELECT 'Alibaba', city_id, 6, 1999 FROM City WHERE city_name = 'Hangzhou';
INSERT INTO Company (name, hq_city, ceo_id, founding_year)
SELECT 'Samsung Electronics', city_id, 7, 1969 FROM City WHERE city_name = 'Suwon';
INSERT INTO Company (name, hq_city, ceo_id, founding_year)
SELECT 'Kering SA', city_id, 8, 1963 FROM City WHERE city_name = 'Paris';
"""
cur.execute(insert_data)
conn.commit()
print(" All seed data inserted successfully.")

InterfaceError: cursor already closed

In [34]:
def show(sql):
    df = pd.read_sql(sql, conn)
    print(df.to_string(index=False))
print("\nCountries:")
show("SELECT * FROM Country ORDER BY country_name;")

print("\nStates:")
show("""
    SELECT s.state_id, s.state_name, c.country_name
    FROM State s
    JOIN Country c ON s.country_id = c.country_id
    ORDER BY c.country_name, s.state_name;
""")

print("\nCities:")
show("""
    SELECT ci.city_name,
           s.state_name,
           c.country_name
    FROM City ci
    LEFT JOIN State   s ON ci.state_id   = s.state_id
    JOIN      Country c ON ci.country_id = c.country_id
    ORDER BY c.country_name, s.state_name NULLS LAST, ci.city_name;
""")

print("\nCEOs:")
show("SELECT * FROM CEO ORDER BY ceo_id;")

print("\nCompanies:")
show("""
    SELECT co.name AS company,
           ci.city_name,
           c.country_name,
           ce.first_name || ' ' || ce.last_name AS ceo,
           co.founding_year
    FROM Company co
    JOIN City    ci ON co.hq_city = ci.city_id
    JOIN Country c  ON ci.country_id = c.country_id
    JOIN CEO     ce ON co.ceo_id = ce.ceo_id
    ORDER BY co.name;
""")


Countries:


C:\Users\User\AppData\Local\Temp\ipykernel_37580\1194146016.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


DatabaseError: Execution failed on sql 'SELECT * FROM Country ORDER BY country_name;': relation "country" does not exist
LINE 1: SELECT * FROM Country ORDER BY country_name;
                      ^


In [24]:
def show(table_sql: str):
    df = pd.read_sql(table_sql, conn)
    print(df.to_string(index=False))

show("SELECT * FROM ceo ORDER BY ceo_id;")

 ceo_id     first_name last_name  birth_year                          birth_place  start_year
      1            Tim      Cook        1960                      Mobile, AL, USA        2011
      2         Arvind   Krishna        1962 West Godavari, Andhra-Pradesh, India        2020
      3           Andy     Jassy        1968                   Scarsdale, NY, USA        2021
      4           Mary     Barra        1961                   Royal Oak, MI, USA        2014
      5      Kenichiro   Yoshida        1959                      Kumamoto, Japan        2018
      6         Daniel     Zhang        1972                      Shanghai, China        2015
      7 Francois-Henri   Pinault        1962                       Rennes, France        2005
      8            Han  Jong-hee        1962                   Seoul, South Korea        2022
      9           Doug  McMillon        1966                     Memphis, TN, USA        2014


C:\Users\User\AppData\Local\Temp\ipykernel_37580\3150158910.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(table_sql, conn)


In [25]:
print("\nQ1: First names of CEOs whose company headquarters are located in the USA:")
show("""
    SELECT DISTINCT c.first_name
    FROM CEO c
    JOIN Company co ON co.ceo_id = c.ceo_id
    JOIN City ci ON ci.city_name = co.hq_city
    WHERE ci.country = 'USA'
    ORDER BY c.first_name
""")

print("\nQ2: Headquarters city of the company whose CEO's last name is Yoshida:")
show("""
    SELECT co.hq_city
    FROM Company co
    JOIN CEO c ON c.ceo_id = co.ceo_id
    WHERE c.last_name = 'Yoshida'
""")

print("\nQ3: Founding years of companies whose CEOs were born between 1960 and 1969:")
show("""
    SELECT co.name AS company, co.founding_year
    FROM Company co
    JOIN CEO c ON c.ceo_id = co.ceo_id
    WHERE c.birth_year BETWEEN 1960 AND 1969
    ORDER BY co.name
""")

print("\nQ4: Years of service for the CEO of the company founded in 1908:")
show("""
    SELECT (EXTRACT(YEAR FROM CURRENT_DATE)::INT - c.start_year) AS years_served
    FROM Company co
    JOIN CEO c ON c.ceo_id = co.ceo_id
    WHERE co.founding_year = 1908
""")

print("\nQ5: Age of the CEO whose company's headquarters is in Hangzhou:")
show("""
    SELECT (EXTRACT(YEAR FROM CURRENT_DATE)::INT - c.birth_year) AS ceo_age
    FROM Company co
    JOIN CEO c ON c.ceo_id = co.ceo_id
    WHERE co.hq_city = 'Hangzhou'
""")


Q1: First names of CEOs whose company headquarters are located in the USA:
first_name
      Andy
    Arvind
      Doug
      Mary
       Tim

Q2: Headquarters city of the company whose CEO's last name is Yoshida:
hq_city
  Tokyo

Q3: Founding years of companies whose CEOs were born between 1960 and 1969:
            company  founding_year
             Amazon           1994
              Apple           1976
     General Motors           1908
                IBM           1911
          Kering SA           1963
Samsung Electronics           1969
            Walmart           1962

Q4: Years of service for the CEO of the company founded in 1908:
 years_served
           11

Q5: Age of the CEO whose company's headquarters is in Hangzhou:
 ceo_age
      53


C:\Users\User\AppData\Local\Temp\ipykernel_37580\3150158910.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(table_sql, conn)
C:\Users\User\AppData\Local\Temp\ipykernel_37580\3150158910.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(table_sql, conn)
C:\Users\User\AppData\Local\Temp\ipykernel_37580\3150158910.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(table_sql, conn)
C:\Users\User\AppData\Local\Temp\ipykernel_37580\3150158910.py:2: UserWarning: pandas only 